In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

In [ ]:
# Substitua pelas URLs diretas dos arquivos (ex: links do GitHub Raw ou S3)
URLS = [
    "https://data.insideairbnb.com/united-states/hi/hawaii/2025-09-16/data/listings.csv.gz"
]

current_directory = os.getcwd() 
DESTINATION_DIR = os.path.join(os.path.join(current_directory, '..'), "data") 

# Cria a pasta 'data' se ela não existir
os.makedirs(DESTINATION_DIR, exist_ok=True)

def download_file(url: str, destination_dir: str):
    """Baixa um arquivo de uma URL direta e salva no diretório de destino."""
    
    # Extrai o nome do arquivo da URL
    file_name = url.split('/')[-1]
    destination_path = os.path.join(destination_dir, file_name)

    print(f"Baixando: {file_name}...")

    try:
        # Realiza a requisição HTTP
        response = requests.get(url, stream=True)
        response.raise_for_status() # Verifica se houve erro (404, 500, etc)

        # Escreve o conteúdo no arquivo local
        with open(destination_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        print(f" > Sucesso: {file_name} salvo em {destination_dir}")

    except Exception as e:
        print(f" > ERRO ao baixar {url}: {e}")

# Executa o loop de download
for url in URLS:
    download_file(url, DESTINATION_DIR)

print("\n--- Processo Finalizado ---")

In [ ]:

#ARQUIVO
caminho = '..\\data\\listings.csv.gz'

df = pd.read_csv(caminho, compression='gzip', on_bad_lines='warn')
#df = pd.read_csv(DESTINATION_DIR + 'listings.csv.gz', sep=',', on_bad_lines='warn')

#INSPEÇÃO
print('\nPrimeiras 5 linhas do DataFrame:')
display(df.head())

df.info()

In [ ]:
# tamanho do conjunto de dados
# e identificação de valores ausentes

print(f'O conjunto de dados possui {df.shape[0]} linhas e {df.shape[1]} colunas.')

missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_info = pd.DataFrame({
    'Valores Ausentes': missing_values,
    'Porcentagem (%)': missing_percentage
})

missing_info = missing_info[missing_info['Valores Ausentes'] > 0].sort_values(by='Porcentagem (%)', ascending=False)

print('\nValores ausentes por coluna (e suas porcentagens):')
display(missing_info)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Coluna 'price' para numérico
# Removendo '$' e ',' e convertendo para float
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

print(df['neighbourhood_cleansed'].head())
# Calcular a média do preço por bairro
expensive_neighborhoods = df.groupby('neighbourhood_cleansed')['price'].mean().sort_values(ascending=False)

print('Os 10 bairros mais caros (preço médio):')
print(expensive_neighborhoods.head(10))

plt.figure(figsize=(10, 6))

# Distribuição de preços
sns.histplot(df['price'].dropna(), bins=30, kde=True)
plt.title('Distribuição de Preços')
plt.xlabel('Preço')
plt.ylabel('Contagem')
plt.xlim(0, 8000)
plt.tight_layout()
plt.show()



sns.barplot(x=expensive_neighborhoods.head(10).index, y=expensive_neighborhoods.head(10).values)
plt.title('Top 10 Bairros Mais Caros')
plt.xlabel('Bairro')
plt.ylabel('Preço Médio')
plt.xticks(rotation=45)
plt.ylim(0, 2000)
plt.tight_layout()
plt.show()


In [ ]:
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

#top 10 most expensive neighborhoods for the violin plot
top_10_expensive_neighborhoods = expensive_neighborhoods.head(10).index.tolist()
df_top_10_expensive = df[df['neighbourhood_cleansed'].isin(top_10_expensive_neighborhoods)]

#limit the price
df_top_10_expensive = df_top_10_expensive = df_top_10_expensive[df_top_10_expensive['price'] < 1000]

plt.figure(figsize=(12, 7))
sns.violinplot(x='neighbourhood_cleansed', y='price', data=df_top_10_expensive, order=top_10_expensive_neighborhoods)
sns.set_theme(context='notebook', style='darkgrid', palette='deep', font='sans-serif', font_scale=1, color_codes=True, rc=None)
plt.title('Violin Plot - Preço por Bairro (Top 10 Mais Caros) - Limite $1000')
plt.xlabel('Bairro')
plt.ylabel('Preço')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
#Score relation to neighboorhood

df['review_scores_rating'] = pd.to_numeric(df['review_scores_rating'], errors='coerce')
df['review_scores_rating'] = df['review_scores_rating'].fillna(0)

# Média de scores
average_scores = df.groupby('neighbourhood_cleansed')['review_scores_rating'].mean().sort_values(ascending=False)


top_10_neighborhoods = average_scores.head(10).index.tolist()

plt.figure(figsize=(10, 6))
sns.barplot(x=average_scores.head(10).index, y=average_scores.head(10).values)
plt.title('Média de Scores por Bairro (Top 10)')
plt.xlabel('Bairro')
plt.ylabel('Média de Scores')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

#violin plot
df_top_10_scores = df[df['neighbourhood_cleansed'].isin(top_10_neighborhoods)]

plt.figure(figsize=(10, 6))
sns.violinplot(x='neighbourhood_cleansed', y='review_scores_rating', data=df_top_10_scores, order=top_10_neighborhoods)
plt.title('Violin Plot - Score por Bairro (Top 10)')
plt.xlabel('Bairro')
plt.ylabel('Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()




In [ ]:
#quantidade de 'amenities'

amendidades = df['amenities'] = df['amenities'].astype(str)

print(amendidades.head())



In [ ]:
import ast
from collections import Counter


def parse_amenities_list(amenities_str):
    try:
        return ast.literal_eval(amenities_str)
    except (ValueError, SyntaxError):
        return []

df['parsed_amenities'] = df['amenities'].fillna('[]').apply(parse_amenities_list)


all_amenities = [item for sublist in df['parsed_amenities'] for item in sublist]

amenity_counts = Counter(all_amenities)


print('Contagem das Comodidades (Top 20):')
for amenity, count in amenity_counts.most_common(100):
    print(f'- {amenity}: {count}')

In [ ]:
amenity_counts = Counter(all_amenities)
sns.set_theme()

# top 20 amenities
top_20_amenities = amenity_counts.most_common(50)
amenities_df_top_20 = pd.DataFrame(top_20_amenities, columns=['Amenity', 'Count'])

plt.figure(figsize=(12, 8))
sns.barplot(x='Count', y='Amenity', hue='Amenity', data=amenities_df_top_20.sort_values(by='Count', ascending=False), palette='viridis', legend=False)
plt.title('Top 20 Most Frequent Amenities')
plt.xlabel('Count')
plt.ylabel('Amenity')
plt.tight_layout()
plt.show()




In [ ]:
import ast

def parse_amenities_list(amenities_str):
    try:
        return ast.literal_eval(amenities_str)
    except (ValueError, SyntaxError):
        return []

# Ensure 'parsed_amenities' column exists, re-creating if necessary
# This line is added to handle cases where the kernel state might have been reset or previous cell not executed
if 'parsed_amenities' not in df.columns:
    df['parsed_amenities'] = df['amenities'].fillna('[]').apply(parse_amenities_list)

# Filtrar o DataFrame para acomodações que contêm 'Beach essentials'
df_beach_essentials = df[df['parsed_amenities'].apply(lambda x: 'Beach essentials' in x)]

# Contar o número de acomodações com 'Beach essentials' por bairro
beach_essentials_by_neighborhood = df_beach_essentials['neighbourhood_cleansed'].value_counts().reset_index()
beach_essentials_by_neighborhood.columns = ['Neighborhood', 'Count']

# Selecionar os 10 bairros com mais 'Beach essentials'
top_10_beach_essentials = beach_essentials_by_neighborhood.head(10)

print("Top 10 bairros com mais 'Beach essentials':")
display(top_10_beach_essentials)

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(x='Count', y='Neighborhood', data=top_10_beach_essentials, palette='viridis')
plt.title("Top 10 Bairros com Mais 'Beach essentials'")
plt.xlabel('Número de Acomodações com Beach essentials')
plt.ylabel('Bairro')
plt.tight_layout()
plt.show()

In [ ]:
# bairros com 'beach essentials'

import plotly.express as px
import json
import unicodedata
from urllib.request import urlopen


GEOJSON_HAWAII =  'https://data.insideairbnb.com/united-states/hi/hawaii/2025-09-16/visualisations/neighbourhoods.geojson'

def normalizar(texto: str) -> str:
    return unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('ASCII')


def carregar_geojson(url: str) -> dict:
    with urlopen(url) as response:
        geojson = json.load(response)

    for feature in geojson["features"]:
        nome = feature["properties"].get("name",
                                      feature["properties"].get("NAME",
                                                              feature["properties"].get("neighbourhood", None)))
        if nome is None:
            print(f"Warning: Could not find a suitable name key in properties for a feature. Available keys: {feature['properties'].keys()}")
            nome = "Unnamed Neighborhood"

        feature["properties"]["name"] = normalizar(nome)

    return geojson


def main():

  df_hawaii = beach_essentials_by_neighborhood.copy()
  df_hawaii = df_hawaii.rename(columns={'Neighborhood': 'bairro_nome', 'Count': 'quantidade'})
  df_hawaii['bairro_nome'] = df_hawaii['bairro_nome'].apply(normalizar)

  geojson = carregar_geojson(GEOJSON_HAWAII)

  fig = px.choropleth_mapbox(
      df_hawaii,
      geojson=geojson,
      locations='bairro_nome',
      color='quantidade',
      featureidkey='properties.name',
      mapbox_style='carto-positron',
      zoom=6,
      center={'lat': 21.30694, 'lon': -157.85833},
      title = 'Número de acomodações com "Beach Essentials" por Bairro no Havaí'
      )

  fig.show()

if __name__ == '__main__':
  main()


In [ ]:
import plotly.express as px
import json
import unicodedata
from urllib.request import urlopen

GEOJSON_HAWAII =  'https://data.insideairbnb.com/united-states/hi/hawaii/2025-09-16/visualisations/neighbourhoods.geojson'

def normalizar(texto: str) -> str:
    return unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('ASCII')


def carregar_geojson(url: str) -> dict:
    with urlopen(url) as response:
        geojson = json.load(response)

    for feature in geojson["features"]:
        nome = feature["properties"].get("name",
                                      feature["properties"].get("NAME",
                                                              feature["properties"].get("neighbourhood", None)))
        if nome is None:
            print(f"Warning: Could not find a suitable name key in properties for a feature. Available keys: {feature['properties'].keys()}")
            nome = "Unnamed Neighborhood"

        feature["properties"]["name"] = normalizar(nome)

    return geojson

def main():

  df_hawaii_avg = average_scores.copy()
  #display(df_hawaii_avg)

  df_hawaii_avg = df_hawaii_avg.reset_index().rename(columns={'neighbourhood_cleansed': 'bairro_nome', 'review_scores_rating': 'quantidade'})
  df_hawaii_avg['bairro_nome'] = df_hawaii_avg['bairro_nome'].apply(normalizar)

  geojson = carregar_geojson(GEOJSON_HAWAII)

  fig = px.choropleth_mapbox(
      df_hawaii_avg,
      geojson=geojson,
      locations='bairro_nome',
      color='quantidade',
      featureidkey='properties.name',
      mapbox_style='carto-positron',
      zoom=6,
      center={'lat': 21.30694, 'lon': -157.85833},
      title = 'Score médio de cada bairro no Havaí'
      )

  fig.show()


if __name__ == '__main__':
  main()

# Ver os monopolio



In [ ]:
from scipy import stats

# Separando os grupos
superhost = df[df['host_is_superhost'] == 't']['price'].dropna()
not_superhost = df[df['host_is_superhost'] == 'f']['price'].dropna()

t_stat, p_val = stats.ttest_ind(superhost, not_superhost, equal_var=False)

print(f"P-valor: {p_val}")
# Se p_val < 0.05, rejeitamos H0 e há diferença real.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Limpeza dos Dados
# Transformar '$150.00' em 150.0 (float)
df['price'] = df['price'].astype(float)

# Criar a coluna booleana 'has_wifi'
# O campo amenities geralmente é uma lista em formato de string ou JSON
df['has_wifi'] = df['amenities'].str.contains('Wifi|Wireless Internet', case=False, na=False)

# 2. Filtrar Outliers (Opcional, mas recomendado para preços)
# Vamos focar no "grosso" dos dados (até o percentil 95) para evitar distorções
q_95 = df['price'].quantile(0.95)
df_filtered = df[df['price'] <= q_95]

# 3. Separar os grupos
group_wifi = df_filtered[df_filtered['has_wifi'] == True]['price'].dropna()
group_no_wifi = df_filtered[df_filtered['has_wifi'] == False]['price'].dropna()

# 4. Executar o Teste Estatístico (Mann-Whitney U)
stat, p_value = stats.mannwhitneyu(group_no_wifi, group_wifi, alternative='less')

# 5. Resultados
print(f"Mediana de preço SEM Wi-Fi: {group_no_wifi.median()}")
print(f"Mediana de preço COM Wi-Fi: {group_wifi.median()}")
print(f"P-valor: {p_value:.10f}")

if p_value < 0.05:
    print("\nConclusão: Rejeitamos a hipótese nula. Airbnbs SEM Wi-Fi são significativamente MAIS BARATOS.")
else:
    print("\nConclusão: Não há evidências estatísticas de que a falta de Wi-Fi torne o imóvel mais barato.")

In [ ]:
import ast

# 1. Converte a string de amenities para uma lista real de Python
# Usamos literal_eval porque é mais seguro que o json.loads para o formato do Airbnb
df['amenities_list'] = df['amenities'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

# 2. Explode a coluna: se um anúncio tinha 10 itens, ele vira 10 linhas
df_exploded = df.explode('amenities_list')

# 3. Limpeza básica (remover espaços extras)
df_exploded['amenities_list'] = df_exploded['amenities_list'].str.strip()

# Visualizar as 10 amenities mais frequentes no Havaí
print("Top 10 Comodidades mais comuns:")
print(df_exploded['amenities_list'].value_counts().head(10))

In [ ]:
from scipy import stats

target_amenity = 'Beach access'
df['has_target'] = df['amenities_list'].apply(lambda x: 1 if target_amenity in x else 0)

limit = df['price'].quantile(0.95)
df_test = df[df['price'] <= limit].copy()

precos_com = df_test[df_test['has_target'] == 1]['price'].dropna()
precos_sem = df_test[df_test['has_target'] == 0]['price'].dropna()

t_stat, p_val = stats.ttest_ind(precos_com, precos_sem, equal_var=False)

print(f"\n--- Resultado para: {target_amenity} ---")
print(f"Média COM: ${precos_com.mean():.2f}")
print(f"Média SEM: ${precos_sem.mean():.2f}")
print(f"P-Valor: {p_val:.10f}")

if p_val < 0.05:
    print(f"Significativo! Ter '{target_amenity}' altera o preço médio.")
else:
    print(f"Não significativo. O preço não parece ser afetado apenas por essa amenity.")

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 1. Configuração visual
sns.set_theme(style="white", palette="muted")
plt.figure(figsize=(12, 8))

# 2. Plotar os pontos individuais (cada anúncio)
# Usamos o 'jitter' para que os pontos não fiquem todos em cima da mesma linha
# O 'alpha' deixa os pontos semi-transparentes para vermos a densidade
sns.stripplot(
    data=df_test, 
    x='has_target', 
    y='price', 
    hue='has_target', 
    jitter=0.35, # Controla o quanto os pontos se espalham
    size=3,       # Tamanho dos pontos (reduzimos por ter muitos dados)
    alpha=0.2,    # Transparência (0.1 a 0.3 é bom para muitos dados)
    legend=False
)

# 3. Sobrepor a representação estatística (Média + Intervalo de Confiança)
# Isso foca no resultado que o Teste T avaliou
sns.pointplot(
    data=df_test, 
    x='has_target', 
    y='price', 
    dodge=False, # Mantém centralizado
    join=False,  # Não desenha linha ligando os grupos
    color='black', # Cor contrastante para a estatística
    scale=1.2,     # Tamanho do ponto da média
    errorbar=('ci', 95), # Barra de erro: Intervalo de Confiança de 95%
    capsize=0.1,  # Largura do topo da barra de erro
    markers='D'    # Losango para representar a média
)

# 4. Personalização dos Rótulos
plt.title(f'Distribuição e Média de Preços\nCom vs. Sem "{target_amenity}"', fontsize=16, pad=20)
plt.ylabel('Preço (até percentil 95)', fontsize=12)
plt.xlabel(None)

# Mudar os rótulos do eixo X
plt.xticks([0, 1], [f'Sem {target_amenity}', f'Com {target_amenity}'], fontsize=12)

# Adicionar o valor exato das médias no gráfico para facilitar a leitura
media_sem = df_test[df_test['has_target'] == 0]['price'].mean()
media_com = df_test[df_test['has_target'] == 1]['price'].mean()

plt.text(0, df_test['price'].max() * 0.95, f'Média: ${media_sem:.2f}', ha='center', color='black', fontweight='bold')
plt.text(1, df_test['price'].max() * 0.95, f'Média: ${media_com:.2f}', ha='center', color='black', fontweight='bold')

plt.tight_layout()
plt.show()

    

In [ ]:
# Teste 2: ANOVA - preço médio por ilha/condado
import pandas as pd
from scipy import stats


# Escolher coluna de bairro/neighbourhood
neigh_col = 'neighbourhood_cleansed' if 'neighbourhood_cleansed' in df.columns else ('neighbourhood' if 'neighbourhood' in df.columns else None)
if neigh_col is None:
    raise KeyError('Nenhuma coluna de neighbourhood encontrada no DataFrame.')

# Grupos de interesse (ilhas/condados principais do Havaí)

target_groups = expensive_neighborhoods.head(10)
target_groups = target_groups.index.tolist()  # Extrair os nomes dos bairros/ilhas

# Filtrar apenas para os grupos existentes no dataset
existing_groups = [g for g in target_groups if g in df[neigh_col].unique()]
print('Grupos encontrados no dataset:', existing_groups)

# Verificar se há grupos suficientes para ANOVA
if len(existing_groups) < 2:
    print('Erro: É preciso pelo menos 2 grupos presentes nos dados para executar ANOVA. Verifique a coluna neighbourhood.')
else:
    df_anova = df[df[neigh_col].isin(existing_groups)].copy()
    df_anova['price'] = pd.to_numeric(df_anova['price'], errors='coerce')

    # Remover NaNs e limitar por percentil 95 para reduzir impacto de outliers
    q95 = df_anova['price'].quantile(0.95)
    df_anova = df_anova[df_anova['price'] <= q95]

    # Preparar tuplas (nome, valores) por grupo
    group_tuples = [(g, df_anova[df_anova[neigh_col] == g]['price'].dropna()) for g in existing_groups]

    for g, vals in group_tuples:
        print(f" - {g}: n={len(vals)}, média={vals.mean():.2f}, mediana={vals.median():.2f}")

    # Filtrar grupos com amostras suficientes (>=2) para o teste
    groups_for_test = [vals for g, vals in group_tuples if len(vals) >= 2]
    groups_names_for_test = [g for g, vals in group_tuples if len(vals) >= 2]

    if len(groups_for_test) < 2:
        print('Aviso: Menos de dois grupos com amostras suficientes (>=2) — ANOVA não será executada.')
    else:
        # Executar ANOVA one-way usando apenas grupos válidos
        f_stat, p_val = stats.f_oneway(*groups_for_test)
        print(f"\nANOVA one-way: F = {f_stat:.4f}, p = {p_val:.10f}")
        if p_val < 0.05:
            print('Resultado: Rejeitamos H0 — existe diferença significativa entre as médias dos condados/ilhas.')
        else:
            print('Resultado: Não rejeitamos H0 — não há evidência de diferença nas médias entre os grupos.')

# 1. Configuração visual
plt.figure(figsize=(14, 8))
sns.set_theme()

# 2. Criação do gráfico
# Usaremos o df_anova que você já filtrou e limpou no código anterior
ax = sns.boxplot(
    data=df_anova, 
    x=neigh_col, 
    y='price', 
    order=existing_groups, # Mantém a ordem dos mais caros que você definiu
    palette='viridis',
    hue=neigh_col,
    legend=False
)

# 3. Personalização
plt.title('Distribuição de Preços por Bairro/Ilha (Top 10 mais caros)', fontsize=16)
plt.xlabel('Localização', fontsize=12)
plt.ylabel('Preço (Limitado ao Percentil 95)', fontsize=12)
plt.xticks(rotation=45, ha='right') # Rotaciona os nomes para não sobrepor

# Adicionando uma linha horizontal com a média global para comparação
plt.axhline(df_anova['price'].mean(), color='red', linestyle='--', label='Média Global')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr

# 1. Preparação dos dados (Substitua 'df' pelo seu DataFrame do Inside Airbnb)
# Filtrando apenas as colunas necessárias e removendo valores nulos
df_clean = df[['price', 'reviews_per_month']].dropna()

# Opcional: Remover preços zero ou outliers extremos para não poluir o gráfico
df_clean = df_clean[df_clean['price'] > 0]
df_clean = df_clean[df_clean['price'] < df_clean['price'].quantile(0.95)] # Mantém 95% dos dados

# 2. Execução do Teste de Correlação (Spearman)
corr, p_value = spearmanr(df_clean['price'], df_clean['reviews_per_month'])

print(f"--- Resultado do Teste ---")
print(f"Coeficiente de Correlação: {corr:.4f}")
print(f"P-valor: {p_value:.4e}")

# 3. Interpretação
if p_value < 0.05:
    print("\nConclusão: Existe evidência estatística de correlação.")
    if corr < 0:
        print("A hipótese foi confirmada: A correlação é negativa (mais caro = menos demanda).")
    else:
        print("A correlação é positiva (mais caro = mais demanda).")
else:
    print("\nConclusão: Não há evidência estatística significativa de correlação.")

# 4. Visualização
plt.figure(figsize=(10, 6))
sns.regplot(data=df_clean, x='price', y='reviews_per_month', 
            scatter_kws={'alpha':0.3, 'color': 'gray'}, 
            line_kws={'color':'red'})

plt.title('Relação entre Preço e Demanda (Reviews per Month)')
plt.xlabel('Preço')
plt.ylabel('Reviews por Mês (Proxy de Demanda)')
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# 1. Preparação e Filtro
# Criamos dois grupos distintos: um para casas/aptos e outro para quartos privados
casa_inteira = df[df['room_type'] == 'Entire home/apt']['reviews_per_month'].dropna()
quarto_privado = df[df['room_type'] == 'Private room']['reviews_per_month'].dropna()

# 2. Execução do Teste T (Independent T-Test)
t_stat, p_value = stats.ttest_ind(casa_inteira, quarto_privado, equal_var=False)

print(f"--- Resultado do Teste T ---")
print(f"Média (Casa/Apto Inteiro): {casa_inteira.mean():.2f}")
print(f"Média (Quarto Privado): {quarto_privado.mean():.2f}")
print(f"P-valor: {p_value:.4e}")

# 3. Interpretação
print("\n--- Conclusão ---")
if p_value < 0.05:
    print("Há uma diferença estatisticamente significativa entre os tipos de acomodação.")
    if casa_inteira.mean() > quarto_privado.mean():
        print("Casas/Aptos inteiros têm, em média, mais demanda.")
    else:
        print("Quartos privados têm, em média, mais demanda.")
else:
    print("Não houve diferença significativa o suficiente para afirmar que um tipo atrai mais hóspedes que o outro.")

# 4. Visualização com Boxplot (ideal para comparar categorias)
plt.figure(figsize=(8, 6))
df_plot = df[df['room_type'].isin(['Entire home/apt', 'Private room'])]
sns.violinplot(data=df_plot, x='room_type', y='reviews_per_month', hue='room_type', palette='Set2', legend=False)

plt.title('Distribuição de Demanda por Tipo de Quarto')
plt.xlabel('Tipo de Acomodação')
plt.ylabel('Reviews por Mês')
plt.ylim(0, df_plot['reviews_per_month'].quantile(0.98)) # Limita o eixo Y para ver melhor a "caixa"
plt.show()